# Paper 3 source-family exclusion experiment

This new post hoc experiment removes one attack family from source training AND calibration, refits the classifier, and compares it with a full-training refit. It evaluates the held-out source-test family and all three audited unstratified external targets. It does not recompute stability warnings or establish an independent fifth dataset. No original inputs are overwritten.

Run **Runtime → Run all**. The first cell installs the original estimator versions before they are imported. If the runtime has already imported sklearn/XGBoost with different versions, restart the session before Run all. The notebook mounts Drive automatically and uses the existing Paper3_DataSet folder. No ZIP upload to Colab is required.

The fixed inventory supports 26 dataset-family combinations, 450 fits and 1,800 evaluation checkpoints. This is a substantial CPU run. Each fit and each evaluation is saved separately. After a disconnection, reconnect Drive and run this SAME notebook again; verified checkpoints resume. Do not edit the code or change package versions mid-run. The final small report ZIP downloads automatically even if the run reports INCOMPLETE. Per-flow predictions and model checkpoints stay in Drive.


In [ ]:
%pip -q install numpy==2.1.3 pandas==2.2.3 scipy==1.16.3 scikit-learn==1.6.1 xgboost==3.4.1 joblib==1.6.0 pyarrow==21.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PAPER3_ROOT = Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
if not PAPER3_ROOT.is_dir():
    raise FileNotFoundError(f'Paper 3 folder not found: {PAPER3_ROOT}')


In [ ]:
EXPERIMENT_SOURCE = '"""Post hoc source-family exclusion with fixed external panels. Originals read only."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport json,hashlib,zipfile,shutil,platform,warnings,gc,time\nimport numpy as np,pandas as pd,joblib,sklearn,xgboost\nfrom sklearn.base import clone\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import roc_curve,confusion_matrix,balanced_accuracy_score\n\nPLAN=\'\'\'Post hoc source-family exclusion experiment. Earlier study outcomes are known.\nAll four original datasets, seeds 2027-2031, 49 original predictors and three original classifier configurations are retained. Exact family strings are dataset-specific; no cross-dataset equivalence is assumed. Before fitting, select every family having at least 50 attack rows in original training AND source-test partitions in all five seeds, and at least 50 retained benign and 50 retained attack calibration/training rows after removal in all seeds. Include a newly fitted full-training reference for each source/seed/model. Retain every family eligibility decision, including unsupported families.\nFor each eligible family, remove its attack rows from BOTH source training and source calibration, fit a new median imputer on retained training only, fit a new scaler on retained training only for logistic regression, clone the original estimator configuration and seed, and fit a new classifier. Use remaining calibration only for the original balanced-accuracy threshold rule. Do not use source-test or target outcomes to fit, tune or calibrate. Historical source models and files remain unchanged.\nPrimary diagnostic: recall of the excluded family on the original disjoint source-test partition, compared with the full-training refit on exactly those rows. Retain support counts, source binary confusion counts, all-family source-test recalls and retained known-family recalls. This is source-family novelty within the same capture, not independent-domain family novelty.\nSecondary diagnostic: binary performance on all three audited unstratified external target panels (100000 rows each), using unchanged panels and the newly fitted source model/threshold. Retain confusion counts and descriptive paired differences versus the full-training refit. Exact family strings are not harmonised across external datasets, and their binary target outcomes do not establish recall of a semantically matched unseen target family. No warning scores are recomputed; historical warnings must NOT be assigned to newly fitted models. Thus this experiment is a detector training ablation, not validation of family-specific stability-warning utility.\nNo target adaptation, hyperparameter tuning, class rebalancing, clipping, new significance test or equivalence claim. Sparse families remain in eligibility records rather than silently dropped. Save a model checkpoint before target scoring, then save each evaluation independently; resume only identical software/code/plan/context/input hashes. Validate all source and target inputs before fitting. Hash saved model and prediction files. Store outputs in a new stage; never modify original experiment folders. Report actual fits and evaluations, failures and exclusions. A complete run requires all planned reference/family fits and external evaluations.\n\'\'\'\nSTAGE=\'supplementary_source_family_holdout_20261025\'\nWORK_BASE=Path(\'/content\')\nMODELS=[\'LogisticRegression\',\'RandomForest\',\'XGBoost\'];SEEDS=list(range(2027,2032))\ndef sha(p):\n h=hashlib.sha256()\n with Path(p).open(\'rb\') as f:\n  for b in iter(lambda:f.read(2**20),b\'\'):h.update(b)\n return h.hexdigest()\ndef write(p,v):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_suffix(p.suffix+\'.tmp\');q.write_text(json.dumps(v,indent=2,allow_nan=False)+\'\\n\');q.replace(p)\ndef js(p):return json.loads(Path(p).read_text())\ndef clean(f,features):\n a=f[features].to_numpy(np.float64,copy=True);a[~np.isfinite(a)]=np.nan\n with np.errstate(over=\'ignore\'):a=a.astype(np.float32)\n a[~np.isfinite(a)]=np.nan;return a\ndef keys(f,features):\n a=clean(f,features).astype(\'<f4\');a[a==0]=0.;a[np.isnan(a)]=np.nan\n return np.ascontiguousarray(a).view(np.dtype((np.void,4*len(features)))).ravel()\ndef choose_threshold(y,p):\n fp,tp,th=roc_curve(y,p,drop_intermediate=False);ok=np.isfinite(th)&(th>=0)&(th<=1);th=th[ok];ba=(tp[ok]+1-fp[ok])/2\n ix=np.flatnonzero(np.isclose(ba,ba.max(),atol=1e-12,rtol=0));return float(th[ix[np.argmin(abs(th[ix]-.5))]])\ndef stats(y,pred):\n tn,fp,fn,tp=map(int,np.bincount(2*np.asarray(y,int)+np.asarray(pred,int),minlength=4))\n if min(tn+fp,fn+tp)<=0:raise ValueError(\'Evaluation needs both classes\')\n ba=(tn/(tn+fp)+tp/(tp+fn))/2\n if abs(ba-balanced_accuracy_score(y,pred))>1e-12:raise ValueError(\'Independent BA check failed\')\n return dict(tn=tn,fp=fp,fn=fn,tp=tp,balanced_accuracy=ba,attack_recall=tp/(tp+fn),false_positive_rate=fp/(tn+fp))\ndef attach_metadata(f,meta):\n if not f.row_id.is_unique:raise ValueError(\'Source row IDs not unique\')\n m=meta.set_index(\'row_id\',verify_integrity=True).loc[f.row_id]\n if not np.array_equal(m.Label.to_numpy(),f.Label.to_numpy()):raise ValueError(\'Metadata labels disagree\')\n if m.Attack.isna().any() or m.Attack.astype(str).str.strip().eq(\'\').any():raise ValueError(\'Missing family metadata\')\n out=f.copy();out[\'family\']=m.Attack.astype(str).to_numpy();return out\ndef inventory(frames):\n families=sorted(set().union(*[set(v[\'train\'].loc[v[\'train\'].Label.eq(1),\'family\'])|set(v[\'source_test\'].loc[v[\'source_test\'].Label.eq(1),\'family\']) for v in frames.values()]))\n rows=[];eligible=[]\n for family in families:\n  valid=True\n  for seed,v in frames.items():\n   tr,ca,te=[v[s] for s in [\'train\',\'calibration\',\'source_test\']]\n   nh=lambda x:int((x.Label.eq(1)&x.family.eq(family)).sum())\n   trk=tr[~(tr.Label.eq(1)&tr.family.eq(family))];cak=ca[~(ca.Label.eq(1)&ca.family.eq(family))]\n   support=[int(trk.Label.eq(c).sum()) for c in [0,1]]+[int(cak.Label.eq(c).sum()) for c in [0,1]]\n   ok=nh(tr)>=50 and nh(te)>=50 and min(support)>=50;valid&=ok\n   rows.append(dict(seed=seed,family=family,train_family_rows=nh(tr),calibration_family_rows=nh(ca),test_family_rows=nh(te),retained_train_benign=support[0],retained_train_attack=support[1],retained_cal_benign=support[2],retained_cal_attack=support[3],seed_supported=ok))\n  if valid:eligible.append(family)\n for r in rows:r[\'eligible_all_five_seeds\']=r[\'family\'] in eligible\n return eligible,rows\n\nclass Workspace:\n def __init__(self,root,context,source):\n  import pyarrow,scipy\n  self.root=Path(root);self.context=context;self.local=WORK_BASE/STAGE/\'run_v1\';self.remote=self.root/STAGE/\'run_v1\';self.cache=WORK_BASE/\'paper3_family_input_cache\'\n  self.local.mkdir(parents=True,exist_ok=True);self.cache.mkdir(parents=True,exist_ok=True)\n  self.identity=dict(code_sha256=hashlib.sha256(source.encode()).hexdigest(),plan_sha256=hashlib.sha256(PLAN.encode()).hexdigest(),context_sha256=hashlib.sha256(json.dumps(context,sort_keys=True).encode()).hexdigest(),software=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,xgboost=xgboost.__version__,joblib=joblib.__version__,pyarrow=pyarrow.__version__,scipy=scipy.__version__))\n def upload(self,rel):\n  a=self.local/rel;b=self.remote/rel;b.parent.mkdir(parents=True,exist_ok=True);q=b.with_suffix(b.suffix+\'.tmp\');shutil.copyfile(a,q)\n  if sha(a)!=sha(q):raise IOError(\'Drive copy differs\')\n  q.replace(b)\n def restore(self,rel):\n  a=self.local/rel;b=self.remote/rel\n  if not a.exists() and b.is_file():a.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(b,a)\n  return a\n def setup(self,source):\n  if sklearn.__version__!=\'1.6.1\' or xgboost.__version__!=\'3.4.1\':raise ValueError(\'Original estimator versions required; rerun setup in a fresh runtime\')\n  p=self.restore(\'identity.json\')\n  if p.exists() and js(p)!=self.identity:raise ValueError(\'Existing stage differs in code/software/context. Do not overwrite it.\')\n  write(p,self.identity);write(self.local/\'CONTEXT.json\',self.context)\n  (self.local/\'EXPERIMENT_CODE.py\').write_text(source);(self.local/\'PLAN.md\').write_text(PLAN)\n  for rel in [\'identity.json\',\'CONTEXT.json\',\'EXPERIMENT_CODE.py\',\'PLAN.md\']:self.upload(rel)\n def input(self,rel):\n  h=self.context[\'pins\'][rel];p=self.cache/rel\n  if p.is_file() and sha(p)==h:return p\n  p.parent.mkdir(parents=True,exist_ok=True);q=p.with_suffix(p.suffix+\'.tmp\');shutil.copyfile(self.root/rel,q)\n  if sha(q)!=h:raise ValueError(\'Pinned input differs: \'+rel)\n  q.replace(p);return p\n def cached(self,marker):\n  p=self.restore(marker)\n  if not p.exists():return None\n  r=js(p)\n  if r[\'identity\']!=self.identity:raise ValueError(\'Checkpoint identity differs\')\n  for rel,h in r[\'outputs\'].items():\n   q=self.restore(rel)\n   if not q.is_file() or sha(q)!=h:raise ValueError(\'Checkpoint output differs: \'+rel)\n  return r[\'details\']\n def finish(self,marker,files,record):\n  write(self.local/marker,dict(identity=self.identity,outputs={r:sha(self.local/r) for r in files},details=record))\n  for rel in files:self.upload(rel)\n  self.upload(marker)\n def package(self,report):\n  write(self.local/\'verification_summary.json\',report)\n  files=[p for p in self.local.rglob(\'*\') if p.is_file() and p.suffix not in [\'.parquet\',\'.joblib\',\'.zip\',\'.tmp\'] and p.name!=\'report_hashes.json\']\n  write(self.local/\'report_hashes.json\',{str(p.relative_to(self.local)):sha(p) for p in files});files.append(self.local/\'report_hashes.json\')\n  p=self.local/(STAGE+\'_report_\'+datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S_%f\')+\'.zip\')\n  with zipfile.ZipFile(p,\'x\',zipfile.ZIP_DEFLATED) as z:\n   for f in files:z.write(f,str(f.relative_to(self.local)))\n  try:self.upload(\'verification_summary.json\');self.upload(\'report_hashes.json\');shutil.copyfile(p,self.remote/p.name)\n  except Exception as e:print(\'Drive report copy failed; download local report:\',str(e))\n  print(json.dumps(report,indent=2));print(\'Results:\',self.remote);return p\n\ndef run(root,context,source):\n w=Workspace(root,context,source);report=dict(status=\'RUNNING\',started_utc=datetime.now(timezone.utc).isoformat(),models_completed=0,evaluations_completed=0,new_fits_this_session=0)\n records=[];evaluations=[];families=[];allframes={}\n try:\n  w.setup(source);features=js(w.input(\'Paper3_Materialized/feature_schema_v1.json\'))[\'predictors_in_order\']\n  if len(features)!=49:raise ValueError(\'Expected49features\')\n  # Preflight every pinned source/model/target input before any new fit.\n  for i,rel in enumerate(context[\'pins\']):\n   w.input(rel)\n   if i%25==0:print(\'Verified inputs\',i+1,\'/\',len(context[\'pins\']),flush=True)\n  for ds in context[\'datasets\']:\n   # Locate the exact historical metadata checkpoint, never pick the latest arbitrarily.\n   rc=context[\'recovery\'][ds];parent=w.root/\'supplementary_raw_metadata_recovery_20261024\'\n   matches=[p for p in parent.glob(\'*/\'+ds+\'/complete.json\') if sha(p)==rc[\'marker_sha256\']]\n   if len(matches)!=1:raise ValueError(\'Expected one exact metadata checkpoint for \'+ds)\n   marker=js(matches[0]);rel=rc[\'lookup_relative\'];mp=matches[0].parent/rel\n   if sha(mp)!=rc[\'lookup_sha256\']:raise ValueError(\'Metadata hash differs\')\n   meta=pd.read_parquet(mp);frames={}\n   for seed in SEEDS:\n    frame={s:attach_metadata(pd.read_parquet(w.input(f\'Paper3_Sanitized/{ds}/seed_{seed}/{s}.parquet\')),meta) for s in [\'train\',\'calibration\',\'source_test\']}\n    for a,b in [(\'train\',\'calibration\'),(\'train\',\'source_test\'),(\'calibration\',\'source_test\')]:\n     if np.intersect1d(keys(frame[a],features),keys(frame[b],features)).size:raise ValueError(\'Source partition signature overlap\')\n    frames[seed]=frame\n   eligible,rows=inventory(frames)\n   if eligible!=context[\'eligible_families\'][ds]:raise ValueError(\'Family eligibility differs from recovered inventory\')\n   families +=[dict(source_dataset=ds,**r) for r in rows];allframes[ds]=frames\n  pd.DataFrame(families).to_csv(w.local/\'family_eligibility.csv\',index=False);w.upload(\'family_eligibility.csv\')\n  planned=sum((1+len(context[\'eligible_families\'][ds]))*len(SEEDS)*len(MODELS) for ds in context[\'datasets\'])\n  expected_evaluations=planned*len(context[\'datasets\'])\n  write(w.local/\'execution_plan.json\',dict(planned_fits=planned,planned_evaluations=expected_evaluations,eligible_families=context[\'eligible_families\']));w.upload(\'execution_plan.json\')\n  print(\'Planned fits:\',planned,\'Evaluations:\',expected_evaluations,flush=True)\n  for ds in context[\'datasets\']:\n   for seed in SEEDS:\n    v=allframes[ds][seed];tr,ca,te=[v[s] for s in [\'train\',\'calibration\',\'source_test\']]\n    for name in MODELS:\n     original=joblib.load(w.input(f\'Paper3_Source_Models/{ds}/seed_{seed}/{name}_model.joblib\'))\n     if original[\'dataset\']!=ds or int(original[\'seed\'])!=seed or original[\'model_name\']!=name or list(original[\'features\'])!=features:raise ValueError(\'Historical model identity differs\')\n     for family in [None]+context[\'eligible_families\'][ds]:\n      token=\'full_reference\' if family is None else \'exclude_\'+hashlib.sha256(family.encode()).hexdigest()[:16]\n      folder=f\'jobs/{ds}/seed_{seed}/{name}/{token}\';marker=folder+\'/fit.json\';record=w.cached(marker)\n      if record is None:\n       start=time.monotonic();train=tr if family is None else tr[~(tr.Label.eq(1)&tr.family.eq(family))];cal=ca if family is None else ca[~(ca.Label.eq(1)&ca.family.eq(family))]\n       if family is not None and ((train.Label.eq(1)&train.family.eq(family)).any() or (cal.Label.eq(1)&cal.family.eq(family)).any()):raise ValueError(\'Excluded family present in fitting/calibration\')\n       im=SimpleImputer(strategy=\'median\');xt=im.fit_transform(clean(train,features)).astype(np.float32);xc=im.transform(clean(cal,features)).astype(np.float32);scaler=None\n       if name==\'LogisticRegression\':scaler=StandardScaler();xt=scaler.fit_transform(xt).astype(np.float32);xc=scaler.transform(xc).astype(np.float32)\n       if xt.shape[1]!=49 or not np.isfinite(xt).all() or not np.isfinite(xc).all():raise ValueError(\'Invalid transformed features\')\n       est=clone(original[\'model\'])\n       with warnings.catch_warnings(record=True) as warns:\n        warnings.simplefilter(\'always\');est.fit(xt,train.Label.to_numpy(np.int8))\n       if any(issubclass(x.category,sklearn.exceptions.ConvergenceWarning) for x in warns):raise ValueError(\'Convergence warning: do not omit failed fit\')\n       cp=est.predict_proba(xc)[:,1]\n       if not np.isfinite(cp).all() or ((cp<0)|(cp>1)).any():raise ValueError(\'Invalid calibration probabilities\')\n       th=choose_threshold(cal.Label.to_numpy(np.int8),cp)\n       bundle=dict(model=est,imputer=im,scaler=scaler,threshold=th,features=features)\n       p=w.local/folder;p.mkdir(parents=True,exist_ok=True);joblib.dump(bundle,p/\'model.joblib\',compress=3)\n       record=dict(source_dataset=ds,seed=seed,model=name,condition=token,excluded_family=family,training_rows=len(train),calibration_rows=len(cal),threshold=th,training_row_ids_sha256=hashlib.sha256(train.row_id.to_numpy(\'<i8\').tobytes()).hexdigest(),calibration_row_ids_sha256=hashlib.sha256(cal.row_id.to_numpy(\'<i8\').tobytes()).hexdigest(),excluded_family_training_rows=0 if family is not None else None,excluded_family_calibration_rows=0 if family is not None else None,elapsed_seconds=time.monotonic()-start,warnings=[str(x.message) for x in warns])\n       w.finish(marker,[folder+\'/model.joblib\'],record);report[\'new_fits_this_session\']+=1;del xt,xc,cp,est,bundle;gc.collect()\n      records.append(record);report[\'models_completed\']=len(records)\n      bundle=None\n      for target in [ds]+[x for x in context[\'datasets\'] if x!=ds]:\n       emarker=folder+\'/eval_\'+target+\'.json\';ev=w.cached(emarker)\n       if ev is None:\n        if bundle is None:bundle=joblib.load(w.restore(folder+\'/model.joblib\'))\n        if target==ds:f=te;truth=te.Label.to_numpy(np.int8)\n        else:\n         prefix=f\'supplementary_natural_prevalence_20261024/run_v1/jobs/{ds}__to__{target}/seed_{seed}\'\n         f=pd.read_parquet(w.input(prefix+\'/features.parquet\'));lab=pd.read_parquet(w.input(prefix+\'/labels.parquet\'))\n         if not f.row_id.equals(lab.row_id) or len(f)!=100000:raise ValueError(\'Target panel alignment/size differs\')\n         truth=lab.Label.to_numpy(np.int8)\n        xx=bundle[\'imputer\'].transform(clean(f,features)).astype(np.float32)\n        if bundle[\'scaler\'] is not None:xx=bundle[\'scaler\'].transform(xx).astype(np.float32)\n        prob=bundle[\'model\'].predict_proba(xx)[:,1]\n        if not np.isfinite(prob).all() or ((prob<0)|(prob>1)).any():raise ValueError(\'Invalid evaluation probabilities\')\n        pred=(prob>=bundle[\'threshold\']).astype(np.int8)\n        ev=dict(source_dataset=ds,target_dataset=target,seed=seed,model=name,condition=token,excluded_family=family,rows=len(f),**stats(truth,pred))\n        if target==ds:\n         familyrows=[]\n         for ff in sorted(te.loc[te.Label.eq(1),\'family\'].unique()):\n          mask=(te.Label.eq(1)&te.family.eq(ff)).to_numpy();n=int(mask.sum());det=int(pred[mask].sum());familyrows.append(dict(family=ff,support=n,detected=det,missed=n-det,recall=det/n,excluded_from_fit=ff==family))\n         ev[\'source_test_family_recalls\']=familyrows\n        rel=folder+\'/predictions_\'+target+\'.parquet\';pd.DataFrame(dict(row_id=f.row_id.to_numpy(),y_true=truth,probability_attack=prob,y_pred=pred)).to_parquet(w.local/rel,index=False)\n        w.finish(emarker,[rel],ev);del xx,prob,pred,f;gc.collect()\n       evaluations.append(ev);report[\'evaluations_completed\']=len(evaluations)\n      write(w.local/\'progress.json\',report);w.upload(\'progress.json\')\n      print(\'Completed models\',len(records),\'/\',planned,\'| evaluations\',len(evaluations),\'|\',ds,seed,name,token,flush=True)\n      del bundle;gc.collect()\n  flat=[{k:v for k,v in r.items() if k!=\'source_test_family_recalls\'} for r in evaluations];scores=pd.DataFrame(flat)\n  refs=scores[scores.condition.eq(\'full_reference\')].set_index([\'source_dataset\',\'target_dataset\',\'seed\',\'model\'])\n  scores[\'ba_change_vs_full_refit\']=[float(r.balanced_accuracy-refs.loc[(r.source_dataset,r.target_dataset,r.seed,r.model),\'balanced_accuracy\']) for _,r in scores.iterrows()];scores.to_csv(w.local/\'scores.csv\',index=False);w.upload(\'scores.csv\')\n  famrows=[]\n  for r in evaluations:\n   for f in r.get(\'source_test_family_recalls\',[]):famrows.append({k:r[k] for k in [\'source_dataset\',\'seed\',\'model\',\'condition\',\'excluded_family\']}|f)\n  fam=pd.DataFrame(famrows);ref=fam[fam.condition.eq(\'full_reference\')].set_index([\'source_dataset\',\'seed\',\'model\',\'family\']).recall\n  fam[\'recall_change_vs_full_refit\']=[float(r.recall-ref.loc[(r.source_dataset,r.seed,r.model,r.family)]) for _,r in fam.iterrows()];fam.to_csv(w.local/\'source_family_scores.csv\',index=False);w.upload(\'source_family_scores.csv\')\n  pd.DataFrame(records).to_csv(w.local/\'fit_records.csv\',index=False);w.upload(\'fit_records.csv\')\n  if len(records)!=planned or len(evaluations)!=expected_evaluations:raise ValueError(\'Incomplete experiment grid\')\n  report.update(status=\'PASS\',planned_models=planned,planned_evaluations=expected_evaluations,eligible_dataset_family_combinations=sum(map(len,context[\'eligible_families\'].values())),completed_utc=datetime.now(timezone.utc).isoformat(),no_new_warning_validation=True,no_independent_new_dataset=True)\n except Exception as e:report.update(status=\'INCOMPLETE\',error_type=type(e).__name__,error=str(e));print(\'STOPPED:\',type(e).__name__,str(e),flush=True)\n return w.package(report)\n'
CONTEXT = {'datasets': ['NF_BoT_IoT_v3', 'NF_CSE_CIC_IDS2018_v3', 'NF_ToN_IoT_v3', 'NF_UNSW_NB15_v3'], 'pins': {'Paper3_Materialized/feature_schema_v1.json': '67d61eebb3377e3053f943f1e4c031237b3ef35dd06296d8538fe552b16be0c6', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/calibration.parquet': '628a07c66ee41949f469ddfdb0c5ad0d48ef867a37b3fbd07d876e8bbdc73ce5', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/source_test.parquet': '2b0bc7cc666945574a29aa13f97803fb587e4c7066af44350508c840f08de1ce', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/train.parquet': '14a818f65a40ad78b82cd72aea23e7df85b53681af0760ece1d549bf2de09af9', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/calibration.parquet': '0495a48fba16e3a5c2b64949ce850523e6a11b4b968f35bb447fefef78aaf1d0', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/source_test.parquet': '49de37f151f2cda111398b8bf75ce7e9d59d26276ad44f66ca73ec000d83f74d', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/train.parquet': 'c59ab8ae48e7177a35e16c9fe12ecba706a480322f9a7077b470b470260cd232', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/calibration.parquet': 'e662915a0340d241ddc91f5347a937ba5817e4e95a4eec18083eb408df395739', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/source_test.parquet': '7fff718f64a3d216e0b00d5d2e76ef9a98edc480661de9c894ead197ae4e1ea6', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/train.parquet': 'd0cc1d0efa9b7b37fb2cd2d6df1fbfbf7def5af8144a6b6e6198c96d7597d0d3', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/calibration.parquet': '2266fcc8e6b125ff9a5ee1334f45d1dc22adc4dd303548a2ce1e5cc9782af325', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/source_test.parquet': '3dc968ed856c70e5569949dcb7690bdb2c488b0581063d4b6d033ff4a7b1f11a', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/train.parquet': 'b5bf843f2a50eb9c35d75a1d2ea6842d27a532652372d880142664b1fa196c27', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/calibration.parquet': '3e947cae3cdf051ba2e5453830348c390522fa316fd7792301909b3d61f54904', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/source_test.parquet': 'f198f84872382e70f1044bcdcad9e3ae3bf2d6f633343b2ab413ec0c2b4fe2a7', 'Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/train.parquet': '9f9a950c33a20f7306c01e673e0df1e2252cd92ea2b5d1f6c741f13d485dc79b', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/calibration.parquet': '2885e67e88dde9d5488c94af486b2bf1d13d1d3cb090b481f116960c05db8c85', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/source_test.parquet': '12200066b125db69e51fc58043543942162a591834a38e0a69d9e26d42d2bb07', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/train.parquet': '27ce500af0b403948b55fda28ea87fa9215a196efad4c710e99fa4643e873736', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/calibration.parquet': '931b742549a2d9980c938d591a24282706069c8ac90e298828244dcec9d00866', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/source_test.parquet': '9c4c427f26b2a2c53ef5d447ad2515d59b1acdacbd38cb7b936dd8e1a6e0b292', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/train.parquet': 'ddba8ea1bb07812b3fd43b506992e7ec537b4941f7726faec0a41f666a8b9f93', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/calibration.parquet': '81296a4e962dc565b95183c5e9c674bc5beb25f88a54ef9ffab00f2730090b0b', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/source_test.parquet': 'ea8de15a143e2c2d467d984c24a934c2ee8c0a71f8bf9048437fe7d35472137a', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/train.parquet': '84ec860b817718b14417af7aefbe8dd53fb9180aa9945f7d576404a94dc609d4', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/calibration.parquet': '7b4f25de62976590ac5cb1397f774f96f7706254fc42cf908807cf313ef00787', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/source_test.parquet': '3b20af47e8fabbc1b6d4c575eee52309ac71ca3be49302fe1d6bf1285b7dc515', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/train.parquet': '61a82c0e5875db2b0a3765c2061dd391b68e32133eec447ed1b8ad626ccd365c', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/calibration.parquet': '6f3921c5882712e8a7f1afcbea6051d968ef7f575c0501d0db96bec8f2520fd5', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/source_test.parquet': '86345d53c306ad04e39285c811fb04d8d8b97130ee7dd61d6d00ac465882d742', 'Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/train.parquet': 'efdf6b1ad2a081baedb8eb1c663ab19c99b13e819d0c8addcac06738b4fa94cc', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/calibration.parquet': 'b0c631594763c8f65787785f9ea4966bfb81c32f07dcebd93a6be82c533cfa32', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/source_test.parquet': 'd95e3fb3c7d0f9a256736a09f6565114786aa34178934c4dc895a8a1d37c993c', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/train.parquet': '5535a60be84cd758ca9362e7045246aaa1face3054df80e9a4a7325d003eec5b', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/calibration.parquet': '5e042bad2ac3da65b876b4ffcb132be46b2f93113f7da9e2affec9bd313ffc4d', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/source_test.parquet': 'e81625480f3fd997b4fc4a52722786922db386fb84b2f5d4fefa947d83ac2e1f', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/train.parquet': '5b61026d0ca14abb02534d29758999e569a7dda65413a2974a4ab21a4e1f2aee', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/calibration.parquet': '2e76b4fcf60391167278eabe2e226e39f1d1a14e39443ee2313313bd213d20be', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/source_test.parquet': '041e23affe790efd8d58587047184c81b85074f5cfe57a0754398116c45b5af1', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/train.parquet': '1c27b7b6c76c58ff8ee663247b846c7a72f2ba36da30580af7c504ef29b4641e', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/calibration.parquet': 'bd5d8b3f3026062f0d076179ff1bcf02dc1df907aae401de36e47ee0cff1569b', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/source_test.parquet': 'fac85d8b954af6b20b529feef19e9f84569c7884321cff78c06ad8e19c84cdfa', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/train.parquet': 'b71b51764b0ce814d8b831a9241637896e2055009c77975849c830edfd3904f2', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/calibration.parquet': '64d1555d8f9e5f22f22cc43cdfa740cef8956a9474ce0e590137cce7e4c0f0e6', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/source_test.parquet': '3487e8b958fb92e288ce62ebf3979ed63da7efadb765026da9c88d6fb8e31e5f', 'Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/train.parquet': '8db5c88c5f4dc288cb3674987c605a99816a122303a22b33eab13c0915c199bd', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/calibration.parquet': 'e30e4d663acb8c82d94f48503e20fc6dcf888c13a1b7faef1707b3f8854af32b', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/source_test.parquet': 'd7bd279fa1a0ddac02d9397fd68abdea33737f7204ec5dd7e567c243d65f44c8', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/train.parquet': 'f732bfba933d1256652544886813c020ee2309cac9181bada34674dcf5626119', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/calibration.parquet': '4e4deba40108efe46df0785c2d40a0d7a45cab50c6c40e01826bda08743b1cee', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/source_test.parquet': '96bab17b4847200c038c3e7c78f408558721f0842a06cf71cdcc26c1217b7a55', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/train.parquet': '5ffa313db0ffd0e01c122cc60397253079f5cecabf355ce48610222410d0764d', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/calibration.parquet': 'e1b7605bfe97c28f49f228b19ccf5c92724c88e951905c4a5e6510cca9c0f773', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/source_test.parquet': '5ef2fa26656225c83c6ab8a91d3d6dd9b2dfbe97228a760a35e9221042364150', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/train.parquet': 'd69b09bc9a9faf8dde11b233e208579f1204412328baa9e0a32a9bf15c107fb7', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/calibration.parquet': 'a19de0f4fc3db885b282424d5e42a247828f008479886a9e134263c6c354c6ed', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/source_test.parquet': 'f1a17d74421e102eb0cf4a214949147ad64af206df4b52f8a528e3181578c9a8', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/train.parquet': '52b7f7c6907338770042bdc7605b2e5c59e6143716eaae8454b67c8d8c683a7b', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/calibration.parquet': '42b84079769948fac863ffb2504851b299bd7b404495310be3aa6a4f24137250', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/source_test.parquet': 'c9ff9f2791f5f1aa027c6ba44c95d74e1613c8649e1076e0064eaed033d7819c', 'Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/train.parquet': '867abf09516e6bd6814912870be39da8c5bd4261d538c9cba756aa0440ced14a', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/LogisticRegression_model.joblib': '945eff37dccb31c89f5af1e716de8dc251e51b1162bec3434821785d519b4d63', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/RandomForest_model.joblib': '35091e1a0f65016d10b267bc4343695d74272a8e13d4a57924e943dc49019038', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/XGBoost_model.joblib': 'e4da21653769fc8c3de80d80e22cdf54d519054c233aae48bd46a789fceb21b0', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/LogisticRegression_model.joblib': '37fe2ab4ed025a0ab3baea97ccbbd677ead47da05144f064a69e2315888d0c3a', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/RandomForest_model.joblib': '353c25340ef656b47574f02dcf27e2775a3fc08e0a4a65f967bae7aaa3bea552', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/XGBoost_model.joblib': '3f25abce8127346480040c82de421ed0b5e836ab8bff6401e95fd45a196fcfc1', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/LogisticRegression_model.joblib': '1a55641b16bd3a793e37502da491ed5f1b8f7eddada3200f9bbd6b4a8c33f9ed', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/RandomForest_model.joblib': '113f2c0ea1f2d9936615219c304b220c3b59dc9d30bdae541e1644135569e2a7', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/XGBoost_model.joblib': 'd621e20a0047102e6beadcf2359c73957d0d045518a1f211d6378e3b2a3d120c', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/LogisticRegression_model.joblib': 'cd5826138e28c8cc60837649687452334dd9abc1c07f7b613c361740b53e323c', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/RandomForest_model.joblib': '72a9e4159843a11c28cf3d2a5e707185ea96b1825382fecf60fdd64d605d9d2b', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/XGBoost_model.joblib': '1a8652ca48ff0584afaee0070ba6ee97dfe4ce41868d07f239f597fef6f405e1', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/LogisticRegression_model.joblib': 'c023f8bcf7e919caa20a6331dac32813c3382eea6a7d8d001881a86017228664', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/RandomForest_model.joblib': 'b41b0e84d3703a2ba76c68e8582dd93a4b04b8a68caf314e44f0671b3d748e53', 'Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/XGBoost_model.joblib': '5cee9d63e801d752ed41123779ae52e1cae66fedfeea39e7c14084fa8e60dfa8', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/LogisticRegression_model.joblib': 'a4f5f9eeebd98df95b0126285e1e51340d1f4c540b0291e9e40920562f61e731', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/RandomForest_model.joblib': '4910110ddf11f388cef84529e9f4a0beb4840d5b86e78b03c32b3f93764a78b5', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/XGBoost_model.joblib': '27e79a0c9635c86cdd644acef41d2219e7bcc6ad13a239809f3ea2df6034889d', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/LogisticRegression_model.joblib': '5158c7538c01692a3b01af82d55563cdd5311be203443c69d42688f3bb1d14f8', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/RandomForest_model.joblib': '7a3f03d67c9cc2040cd23eca68b8ca4c83f1429689c74ae0141614f1d0826003', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/XGBoost_model.joblib': 'fd28059501c2386104073399027675abf5d671ff08b877e82083584bd7846c46', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/LogisticRegression_model.joblib': 'bdcff4120cc1decba2027880b9e23076a42e6ee409d2c94ce72ebe05c8f840d7', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/RandomForest_model.joblib': '5bb2a82b08f8779be40ce3d2744df5301a798c59799ffc77d79b0971c8b8ced5', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/XGBoost_model.joblib': '3bb66f3208959b8f6f76bff7821dab2c32cc5103550f3af91649488b44ae62ad', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/LogisticRegression_model.joblib': '21f1eb21962f0b449841533f2b75ec310d0cd03f110254d0b0077a6fd3979336', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/RandomForest_model.joblib': 'f162952b80c4968d7e3ada1f614f65a9d4f72319126628f4478e8afad342931a', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/XGBoost_model.joblib': '38e70ca3f408d03050eaf2c5a3a9ce3033b4f5d4df717d8ec2141f33ddd7b3f0', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/LogisticRegression_model.joblib': 'c875c70941c212657877e17702c221e09903542aae3d7d3a9818df484ecfc911', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/RandomForest_model.joblib': '35ed86f51d563151095ee4b1ed6b4b856fe3197b93e6d6205acea9c28a363595', 'Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/XGBoost_model.joblib': '0a4e103281f1c550db544fe38076237ee31c64dd7629c1b3012cd8a4ee4c82f5', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/LogisticRegression_model.joblib': 'a7fb4eb4aa36cb60a88f0b11c7bcd0ad7f9b6afaeb95a81677526b9eb0c0cc2f', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/RandomForest_model.joblib': '01ae3c4dddf9191e09fdf08f5abc9a1b23a5063eecda240af1a1c8ef7a9e4c2b', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/XGBoost_model.joblib': '4f7283ec182b714a0915aa2afa9d8035b6e801c14bf07474b774a7e52d361bb7', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/LogisticRegression_model.joblib': 'e586884c687edfaf729b5ba40f8939233b644071eae3b7ebcc03c37dfaa500be', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/RandomForest_model.joblib': '7b07c362d3516f378a12be7cda5016132b4bccd916584c106408709579701039', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/XGBoost_model.joblib': '84bb42b6d9b99f554646131709d6add5036142febc82a77b3400a2c2cbfef5ec', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/LogisticRegression_model.joblib': '7e2a7d33bfd668f3a49f09336903f0ad9180f35fcd096e3d723e3961b77ea87b', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/RandomForest_model.joblib': '0ffbffb9c6ef33a261f36a362a1711e2186d50e3cdea914063414998e0df6fca', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/XGBoost_model.joblib': '22c4e45e4ab22d7f74d51606841acb6a9ed0df0c9118ca00856a5f370e989a9d', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/LogisticRegression_model.joblib': 'ca09a04eb02127cc6e6b37f20576596e09a62960e173cbc05503e9de3f9acd6d', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/RandomForest_model.joblib': 'fc29b0782ed92e848d7b1b234b321a1cf2fc14e027f8e1629c67d74afe8664a2', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/XGBoost_model.joblib': '93621a946c728eb5ebdca8e5281d4b66f3f5df5606275432341e1f325024561a', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/LogisticRegression_model.joblib': '77a4ba42d4b4741f7cef3d7bdacb9cf4ec39ebbd063d84383e913ef58a921b10', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/RandomForest_model.joblib': 'ea57691b6fbd4604df073eb98eb19f1493fe6e5241cc1ee1a4f586e93f448d9d', 'Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/XGBoost_model.joblib': 'f0e2de65596dbfa0a8c82eff142b339902b7326eea3ed33c029dc7ac4c220da3', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/LogisticRegression_model.joblib': '00497b18e2092a684b3caf5dcb2f114493d3a5785aa0f2d894ce54a3de9592fd', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/RandomForest_model.joblib': '16b4d2364e8ae0ac87f2ed26a11f65ca9af74c8ad812e4dd3f2b7d80c11759a8', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/XGBoost_model.joblib': 'e60ea2af1e9c12db6873872440a5212ebadd6dd999fceecbbd288ebf5afd5970', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/LogisticRegression_model.joblib': '97fdde086590003138e08c2a99cfe98cc1db66fca44e37e7ec4306e0605bbe19', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/RandomForest_model.joblib': '0f404163ad9d5d613e6ff2a7ee332d373cce978d7309353b9675625881999a1d', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/XGBoost_model.joblib': '9b6352547581ac0ae0206650dfa5021ae8e52d8aba280597beab47f22653c2e1', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/LogisticRegression_model.joblib': 'ca912b7ce61827961c0bfd1aad2d97cd15fdf4e0cff555d0f0519ec718eb5ca0', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/RandomForest_model.joblib': 'b842e316d837f33022ab62a8721678664f98795a762f1738be7ca911d94ce3c7', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/XGBoost_model.joblib': '6111e854990ffde731254de46325d355fc39117e0dcb74e9b4efc2a5a2f6dfee', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/LogisticRegression_model.joblib': '1be4d05aa1744c18fe9e3ce0e6e0e69af18decb50f4db3e41a30c89078b06489', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/RandomForest_model.joblib': '3f9fc8431086a72182471c447ca260cad85348f99f201c3a06e0a367c9c50b23', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/XGBoost_model.joblib': '2cc9cc98e580475da80449fd666ed39b8d4a9bcd97862abbf7ad669c34ea6c7f', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/LogisticRegression_model.joblib': '5bc903298aa7db93912a712e7e5dc931b376c881cbad19305e74423451e8b9a8', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/RandomForest_model.joblib': 'd85496859efa480e8b1d2f6b52e2db5a957edb98debb4d863389576c58e0b583', 'Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/XGBoost_model.joblib': 'dc7bb42418d40e48066c334fc8c9c3ce2e8a230923ea94208029b4334e10a50e', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/features.parquet': '43651f3b3af6529640e41a9f3172a81e407c9ed22f4e03dc978635f41086e402', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/labels.parquet': 'a7366de9abc83446e5717c967a7863542b2a56ef74250076302d49d092b3aa2c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/features.parquet': '8a7d79958a04d86ee9b6264a45475ee371e00e932809ddfe8ba929b3f75ac88f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/labels.parquet': '781342095c82b6dc2309a69a064f487b966e8c0f59860fe6bdf99bef60962b5f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/features.parquet': '49f975be64fcd97cfab1eedf68e0755aaad7e0b4567796d107815e02d56f3e40', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/labels.parquet': '5decce53c53abdd0c18e15f67fd8567c87394a1c1c346c6f2f0d55c873ca626d', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/features.parquet': '00aee921bff9803e98136daa1c68683848975d69cf288656e57bfabde1fd10ed', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/labels.parquet': '008ea8b1c26b3bf78202650879681e0bacbd259489334b0ef34388395ed5d0b2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/features.parquet': 'f88898bc5473fdcadef10ac9d732fb759f1f4c2e1cb69a51d20aff3f3843459e', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/labels.parquet': '62583c9124da0571fd7e21bf11ddfbb5021ee127ef42ae47a2922971afe5a421', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2027/features.parquet': 'de42b8a3ee096be3c7f1fdff621eb287483ae47182233bd5600a5de76e5cf5b7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2027/labels.parquet': 'bdb11b4c30b1112a0a5433ec93e680925796b36e72081f78c1329a6510c6dfb0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2028/features.parquet': '9b17ebe74bc8723344b31f9e27abc5b6a6dea3bc19a8df679a93b33de9a8ae57', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2028/labels.parquet': 'bdcf3359b2194f520145811fbf41a3fb5fb2eed72f1fb5f4569e820ba3c952a0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2029/features.parquet': 'cfdfa051c3039358be425e7b61f5472de8ec2a919dbe55af40837621d8467dd0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2029/labels.parquet': 'c8ddcc2e8c53f25b52b8524bae7f118ec079d81aa07906144a825cc086e5e1e7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2030/features.parquet': 'cdd00a22480e6d239117328958d3f89f005a36060a09d1ae63642d6d55aaff2b', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2030/labels.parquet': '241aa377e5625012b7945ddcd5543ab6a87cc65a7dfc1fd5342a17cf6ab36448', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2031/features.parquet': 'fd9820f3b0ac411d752c4438ad8d38052cfa423a15c5c82216c7c16d8d2d2d6a', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_ToN_IoT_v3/seed_2031/labels.parquet': 'b115494f886800c45cfc48d165068f880e8de78f524ebaf74ad5bd194ea7b43d', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2027/features.parquet': 'e61aff14e0ef132ae18f66e1da2e48593266618a8d439d2f80cf0a0e175176c6', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2027/labels.parquet': '01fae00bfd430f9f784495c9018e4967be9916b4ea375a5602eafbe2bbbc62ea', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2028/features.parquet': '5975707941a5ff585c012e4f3ba5a444087946261cd43b6f1849ad3acb98a004', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2028/labels.parquet': '018c58592a7f887945cba28053e6c2afbefdd3d83350918579c1f866d7cd0634', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2029/features.parquet': '5e2699fabe32645ff838872d0e3a7aa167b0d15e1a91bbc09e10dd3e4867c0cb', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2029/labels.parquet': 'e82b723040e8444ffb231882dd5e61009fc9a10d71a923622a8175f0618cc0e8', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2030/features.parquet': '6fd9d886e799b99c023feb70b2c7db7968f249e8523f0952613710392f20b5fc', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2030/labels.parquet': 'f11f22c9a23b766d0bc87ac053008b1480ee4b549510706dc3fc933dddc693b0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2031/features.parquet': 'd23e088f40f976e35203d41ea4b5c1156a18b62dd0a0e27cb82e8e55834ea6fa', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_BoT_IoT_v3__to__NF_UNSW_NB15_v3/seed_2031/labels.parquet': 'f56362c3a4227d5f7924d0c8ef92858ce7532da098c80fb50b13d09cfb4f5f4f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2027/features.parquet': 'be0f418f59caeec4de6918eb96072b2be8892fb81e80c348ab8ba735f133abe7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2027/labels.parquet': '4388357edefcea50b6a6a772a97cf62aec90be767e4859626c9b189d19bee0f2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2028/features.parquet': '982a2043c5828d361736df6739201bc386bc851d6022d7548b3c64fa59a7c62a', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2028/labels.parquet': '6eafd69be9866046bd936d7c94555c944d25418461fe0ca35c147feaf6362648', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2029/features.parquet': 'dd7b51d92b06d33abbedded6187f720362652d97e31c0ccdaaa12acfd4473ce2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2029/labels.parquet': 'e0f89574e1fadcfcd0a16050c45b16b75bd42dc6e00158fd954c80c0cfe283af', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2030/features.parquet': 'd840b1afe746dcb25f75e46558192d5449d2c100ceeeea30ceaa97cc87f135d3', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2030/labels.parquet': 'f790dee9a7dd950bc28cae2bd7d9954e103974270c874b190ffdd554e6c409be', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2031/features.parquet': 'e1cdb9ec7b1eaec36b04673465e9c52b67cd368f2ad52bb008b56a62ccc9f939', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_BoT_IoT_v3/seed_2031/labels.parquet': '13e0405608fbef95b58e8a82bf7709747f7a2bdbace5df1273c9583c4cce598c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2027/features.parquet': '08802dfb22b2c1e9f60c30e836c2e3489af51b6beca7cff4c9c8225b3180b345', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2027/labels.parquet': 'c6c7b95c513d42e8dd52ea7ab50b01d4d54ae706ad92cbcd236153d7fc6384e4', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2028/features.parquet': '18d14199065cbbefed333c1b43cddfcc53fbecf12ff28d79915ba841ebd6d807', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2028/labels.parquet': '79e20dde83743182405994f1014750e98aacd1e3ba90bbff3a695dc043f9e9ae', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2029/features.parquet': 'c200aec9d2cf67006730b1ecfeb16c8c46e50397133c1278be52c80252a8ed61', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2029/labels.parquet': '310a4a66d628c46aa8a8df602bc50d291036f515a3c08a35def72b039e2586ad', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2030/features.parquet': 'e4c1a0e369a550ed6fd39157f670fedec1dcea502768bb44818cf4d946b30899', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2030/labels.parquet': '6a24de59bf0c22004801a0b1dc1e9598adb3d9fff34fd793c37ab27b8a532bc3', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2031/features.parquet': '146f43fa7d3671eaa5c990f91bcfaf02d7751dd667621f1d53eecdc34ce11751', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_ToN_IoT_v3/seed_2031/labels.parquet': 'ca5a6a2e95fc1ad496af598a3178409c55edb84a5c0bd2fb87276f36df421e55', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2027/features.parquet': 'e61aff14e0ef132ae18f66e1da2e48593266618a8d439d2f80cf0a0e175176c6', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2027/labels.parquet': '01fae00bfd430f9f784495c9018e4967be9916b4ea375a5602eafbe2bbbc62ea', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2028/features.parquet': '5975707941a5ff585c012e4f3ba5a444087946261cd43b6f1849ad3acb98a004', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2028/labels.parquet': '018c58592a7f887945cba28053e6c2afbefdd3d83350918579c1f866d7cd0634', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2029/features.parquet': '5e2699fabe32645ff838872d0e3a7aa167b0d15e1a91bbc09e10dd3e4867c0cb', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2029/labels.parquet': 'e82b723040e8444ffb231882dd5e61009fc9a10d71a923622a8175f0618cc0e8', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2030/features.parquet': '6fd9d886e799b99c023feb70b2c7db7968f249e8523f0952613710392f20b5fc', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2030/labels.parquet': 'f11f22c9a23b766d0bc87ac053008b1480ee4b549510706dc3fc933dddc693b0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2031/features.parquet': 'd23e088f40f976e35203d41ea4b5c1156a18b62dd0a0e27cb82e8e55834ea6fa', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_CSE_CIC_IDS2018_v3__to__NF_UNSW_NB15_v3/seed_2031/labels.parquet': 'f56362c3a4227d5f7924d0c8ef92858ce7532da098c80fb50b13d09cfb4f5f4f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2027/features.parquet': 'be0f418f59caeec4de6918eb96072b2be8892fb81e80c348ab8ba735f133abe7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2027/labels.parquet': '4388357edefcea50b6a6a772a97cf62aec90be767e4859626c9b189d19bee0f2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2028/features.parquet': '982a2043c5828d361736df6739201bc386bc851d6022d7548b3c64fa59a7c62a', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2028/labels.parquet': '6eafd69be9866046bd936d7c94555c944d25418461fe0ca35c147feaf6362648', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2029/features.parquet': 'dd7b51d92b06d33abbedded6187f720362652d97e31c0ccdaaa12acfd4473ce2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2029/labels.parquet': 'e0f89574e1fadcfcd0a16050c45b16b75bd42dc6e00158fd954c80c0cfe283af', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2030/features.parquet': 'd840b1afe746dcb25f75e46558192d5449d2c100ceeeea30ceaa97cc87f135d3', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2030/labels.parquet': 'f790dee9a7dd950bc28cae2bd7d9954e103974270c874b190ffdd554e6c409be', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2031/features.parquet': 'e1cdb9ec7b1eaec36b04673465e9c52b67cd368f2ad52bb008b56a62ccc9f939', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_BoT_IoT_v3/seed_2031/labels.parquet': '13e0405608fbef95b58e8a82bf7709747f7a2bdbace5df1273c9583c4cce598c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/features.parquet': 'fc5dc091986ae4d9dbb935766db8760e87cbf8c2752ed9dbbbb248d55becd4dd', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/labels.parquet': 'a6fdc42d6d563491133afae7a3137d82ed6b223b8a1b7c0b8ddc7c3979ed0fff', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/features.parquet': 'ec83b77c57dce79dfd08b52ef50e9ab037370a4f24c603d3b91d57d94af7bfcf', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/labels.parquet': '6198c3bddd1e4404fe5b8d9a8e23b1490169d62897dc6235d72c22c4ad5781f9', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/features.parquet': '6025a6a4241a05bdcbe2f714fdb9484979bf5703725b2569335b9b66ee9e1734', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/labels.parquet': 'e18ab3ad2d0f71d6a1457f2b50cbfbfc27cf58ebb60c4811f48349ac7ae05c2c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/features.parquet': '2211928921a65409c832f6a31437b513d042bb4511d501fbe1f676215401f6fa', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/labels.parquet': 'd38c5b5e478b3a721847d1707906de28c463e352473f5f2723258bf22c657121', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/features.parquet': 'c33f1b3f9fc87dcd4f9e0770428114f91848df8415eeb758c8257354df68da76', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/labels.parquet': '9294160d4b623daf487bc877565b84b1a5f51fe490f23273d6c7f5abf228c058', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2027/features.parquet': 'e61aff14e0ef132ae18f66e1da2e48593266618a8d439d2f80cf0a0e175176c6', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2027/labels.parquet': '01fae00bfd430f9f784495c9018e4967be9916b4ea375a5602eafbe2bbbc62ea', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2028/features.parquet': '5975707941a5ff585c012e4f3ba5a444087946261cd43b6f1849ad3acb98a004', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2028/labels.parquet': '018c58592a7f887945cba28053e6c2afbefdd3d83350918579c1f866d7cd0634', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2029/features.parquet': '5e2699fabe32645ff838872d0e3a7aa167b0d15e1a91bbc09e10dd3e4867c0cb', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2029/labels.parquet': 'e82b723040e8444ffb231882dd5e61009fc9a10d71a923622a8175f0618cc0e8', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2030/features.parquet': '6fd9d886e799b99c023feb70b2c7db7968f249e8523f0952613710392f20b5fc', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2030/labels.parquet': 'f11f22c9a23b766d0bc87ac053008b1480ee4b549510706dc3fc933dddc693b0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2031/features.parquet': 'd23e088f40f976e35203d41ea4b5c1156a18b62dd0a0e27cb82e8e55834ea6fa', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_ToN_IoT_v3__to__NF_UNSW_NB15_v3/seed_2031/labels.parquet': 'f56362c3a4227d5f7924d0c8ef92858ce7532da098c80fb50b13d09cfb4f5f4f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2027/features.parquet': 'be0f418f59caeec4de6918eb96072b2be8892fb81e80c348ab8ba735f133abe7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2027/labels.parquet': '4388357edefcea50b6a6a772a97cf62aec90be767e4859626c9b189d19bee0f2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2028/features.parquet': '982a2043c5828d361736df6739201bc386bc851d6022d7548b3c64fa59a7c62a', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2028/labels.parquet': '6eafd69be9866046bd936d7c94555c944d25418461fe0ca35c147feaf6362648', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2029/features.parquet': 'dd7b51d92b06d33abbedded6187f720362652d97e31c0ccdaaa12acfd4473ce2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2029/labels.parquet': 'e0f89574e1fadcfcd0a16050c45b16b75bd42dc6e00158fd954c80c0cfe283af', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2030/features.parquet': 'd840b1afe746dcb25f75e46558192d5449d2c100ceeeea30ceaa97cc87f135d3', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2030/labels.parquet': 'f790dee9a7dd950bc28cae2bd7d9954e103974270c874b190ffdd554e6c409be', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2031/features.parquet': 'e1cdb9ec7b1eaec36b04673465e9c52b67cd368f2ad52bb008b56a62ccc9f939', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_BoT_IoT_v3/seed_2031/labels.parquet': '13e0405608fbef95b58e8a82bf7709747f7a2bdbace5df1273c9583c4cce598c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/features.parquet': '43651f3b3af6529640e41a9f3172a81e407c9ed22f4e03dc978635f41086e402', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2027/labels.parquet': 'a7366de9abc83446e5717c967a7863542b2a56ef74250076302d49d092b3aa2c', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/features.parquet': '8a7d79958a04d86ee9b6264a45475ee371e00e932809ddfe8ba929b3f75ac88f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2028/labels.parquet': '781342095c82b6dc2309a69a064f487b966e8c0f59860fe6bdf99bef60962b5f', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/features.parquet': '49f975be64fcd97cfab1eedf68e0755aaad7e0b4567796d107815e02d56f3e40', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2029/labels.parquet': '5decce53c53abdd0c18e15f67fd8567c87394a1c1c346c6f2f0d55c873ca626d', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/features.parquet': '00aee921bff9803e98136daa1c68683848975d69cf288656e57bfabde1fd10ed', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2030/labels.parquet': '008ea8b1c26b3bf78202650879681e0bacbd259489334b0ef34388395ed5d0b2', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/features.parquet': 'f88898bc5473fdcadef10ac9d732fb759f1f4c2e1cb69a51d20aff3f3843459e', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_CSE_CIC_IDS2018_v3/seed_2031/labels.parquet': '62583c9124da0571fd7e21bf11ddfbb5021ee127ef42ae47a2922971afe5a421', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2027/features.parquet': 'de42b8a3ee096be3c7f1fdff621eb287483ae47182233bd5600a5de76e5cf5b7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2027/labels.parquet': 'bdb11b4c30b1112a0a5433ec93e680925796b36e72081f78c1329a6510c6dfb0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2028/features.parquet': '9b17ebe74bc8723344b31f9e27abc5b6a6dea3bc19a8df679a93b33de9a8ae57', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2028/labels.parquet': 'bdcf3359b2194f520145811fbf41a3fb5fb2eed72f1fb5f4569e820ba3c952a0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2029/features.parquet': 'cfdfa051c3039358be425e7b61f5472de8ec2a919dbe55af40837621d8467dd0', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2029/labels.parquet': 'c8ddcc2e8c53f25b52b8524bae7f118ec079d81aa07906144a825cc086e5e1e7', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2030/features.parquet': 'cdd00a22480e6d239117328958d3f89f005a36060a09d1ae63642d6d55aaff2b', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2030/labels.parquet': '241aa377e5625012b7945ddcd5543ab6a87cc65a7dfc1fd5342a17cf6ab36448', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2031/features.parquet': 'fd9820f3b0ac411d752c4438ad8d38052cfa423a15c5c82216c7c16d8d2d2d6a', 'supplementary_natural_prevalence_20261024/run_v1/jobs/NF_UNSW_NB15_v3__to__NF_ToN_IoT_v3/seed_2031/labels.parquet': 'b115494f886800c45cfc48d165068f880e8de78f524ebaf74ad5bd194ea7b43d'}, 'recovery': {'NF_BoT_IoT_v3': {'marker_sha256': 'f705142b2ab51d9c1ec549e3ebaba4b5a7018a51bc578acf156d3e2835110602', 'lookup_relative': 'attempt_20261024_001733_300411/saved_row_metadata.parquet', 'lookup_sha256': '87cc87a342c865e1221c64e2df36a6396a4bb0606ccc9558a8ca709271377b6c'}, 'NF_CSE_CIC_IDS2018_v3': {'marker_sha256': 'ed227b582cd0c2f3280a806e360ecd88aab60c79e8dad1b5666c92a219a53970', 'lookup_relative': 'attempt_20261024_001929_957898/saved_row_metadata.parquet', 'lookup_sha256': '414053c4b5dbbbabf5a8913dbe05a050b51fa2da7b46b0d2c469a9fc810b66a4'}, 'NF_ToN_IoT_v3': {'marker_sha256': '8ad7fa9d74427dba1edc5e3d6164521fe524e0de52b121cdb04d7dad5e831d05', 'lookup_relative': 'attempt_20261024_002148_810517/saved_row_metadata.parquet', 'lookup_sha256': '29b65ee2fa145c6a73c34f20640d497f48ef56cf13231d96993ac356d7bf2218'}, 'NF_UNSW_NB15_v3': {'marker_sha256': '023f40a0d43ab8c31bdbd6e6527169cdd51dd8015de673af036922a9a9a99bdc', 'lookup_relative': 'attempt_20261024_002432_955014/saved_row_metadata.parquet', 'lookup_sha256': '944e9d10b254469ae08c18ef999c79ec3bec5262b713730c15379d7f940580c6'}}, 'eligible_families': {'NF_BoT_IoT_v3': ['DDoS', 'DoS', 'Reconnaissance'], 'NF_CSE_CIC_IDS2018_v3': ['Bot', 'DDOS_attack-HOIC', 'DDoS_attacks-LOIC-HTTP', 'DoS_attacks-GoldenEye', 'DoS_attacks-Hulk', 'DoS_attacks-SlowHTTPTest', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce'], 'NF_ToN_IoT_v3': ['Backdoor', 'ddos', 'dos', 'injection', 'password', 'scanning', 'xss'], 'NF_UNSW_NB15_v3': ['Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']}}
exec(compile(EXPERIMENT_SOURCE, "paper3_source_family_holdout.py", "exec"))


In [ ]:
report_zip = run(PAPER3_ROOT, CONTEXT, EXPERIMENT_SOURCE)
from google.colab import files
files.download(str(report_zip))


## Return the report

Upload the downloaded `supplementary_source_family_holdout_20261025_report_...zip` to this chat for verification. The result folder is `Paper3_DataSet/supplementary_source_family_holdout_20261025/run_v1`. PASS means the planned fit/evaluation grid completed; it does not imply useful detection performance. Do not replace manuscript results until the report is checked.

The same-source family diagnostic and external binary transfer scores answer different questions. Exact family labels are dataset-specific; this experiment does not claim harmonised unseen attack-family transfer across captures or family-specific warning validation.
